# 01 · Càrrega i neteja

Fase 1 del disseny (`posts/vida-material/index.qmd`, apartat 4). Els quaderns s'executen en ordre:

1. `01_carrega_i_neteja`: descarrega les dades si cal, les neteja i desa `dades/processades/mobils.csv`.
2. `02_classificador_avaria`: assigna el tipus d'avaria a tot el conjunt a partir de MobiFix.
3. `03_B1_es_la_bateria`: quan el símptoma apunta a la bateria, ho és?
4. `04_B2_edat_i_reglament`: a quina edat falla la bateria, comparat amb el reglament.
5. `05_B3_per_que_no_sarregla`: per què les reparacions de bateria acaben sense arreglar.

La neteja és a `../comu.py`, perquè tots els quaderns facin servir la mateixa.

Entorn (una sola vegada, des de l'arrel del repositori):

```bash
python3 -m venv posts/vida-material/_analisi/.venv
posts/vida-material/_analisi/.venv/bin/pip install -r posts/vida-material/_analisi/requirements.txt
```

In [1]:
%run ../scripts/01_descarrega_ora.py

Ja existeix: ora_aggregate.csv
Ja existeix: ora_mobifix.csv


In [2]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

import comu

pd.set_option("display.max_rows", 40)

## Dades en brut

Han de coincidir amb l'apartat 3 de l'entrada: 6.290 mòbils, 1.887 amb edat i una mediana de 4 anys.

In [3]:
brut = comu.carrega_aggregate()
print(f"Mòbils: {len(brut):,}")
print(f"Amb edat: {brut['edat'].notna().sum():,}  ·  mediana: {brut['edat'].median():.0f} anys")
brut["repair_status"].value_counts()

Mòbils: 6,290
Amb edat: 1,887  ·  mediana: 4 anys


repair_status
Fixed          3552
Repairable     1547
End of life     962
Unknown         229
Name: count, dtype: int64

## Neteja

Recompte de registres a cada pas, per informar-ne a l'article.

In [4]:
mobils, recompte = comu.neteja(brut)
recompte

,pas,registres
0,Mòbils,6290
1,Amb estat de la reparació conegut,6061
2,Smartphones (filtre provisional),5652
3,... dels quals amb edat,1733
4,... dels quals amb edat ≤ 15 anys,1694
5,... dels quals a final de vida,855


## Revisió del filtre de smartphones

El filtre és provisional. Cal mirar quines categories del proveïdor s'han tret i quines s'han quedat, i ajustar `NO_SMARTPHONE_CATEGORIA` a `comu.py` si cal.

In [5]:
brut_conegut = brut[brut["repair_status"] != "Unknown"].copy()
brut_conegut["marca"] = brut_conegut["brand"].map(comu.normalitza_marca)
tret = ~brut_conegut.index.isin(mobils.index)
print("Categories tretes:")
display(brut_conegut.loc[tret, "partner_product_category"].value_counts().head(15))
print("Categories que es queden:")
mobils["partner_product_category"].value_counts().head(15)

Categories tretes:


partner_product_category
Computer equipment / phones ~ Mobile phone / cell phone (not smartphone)      183
Elektro divers ~ Telefon                                                      156
Smartphone/Tablet ~ Mobiltelefon                                                9
Smartphone/Tablet ~ Handy                                                       8
Computer equipment / phones ~ Smartphone                                        7
Andet ~ Telefon                                                                 7
Smartphone/Tablet ~ Smartphone                                                  4
Display and sound equipment ~ Mobile phone / cell phone (not smartphone)        4
Mobil/smartphone ~ Telefon                                                      4
Display and sound equipment ~ Smartphone                                        3
Mobile                                                                          3
Mobile ~ Téléphone                                                       

Categories que es queden:


partner_product_category
Mobile                                      1277
Mobile ~ iPhone                              681
Computer equipment / phones ~ Smartphone     674
Smartphone/Tablet ~ Smartphone               638
Mobile ~ Mobile Phone                        390
Mobile ~ Phone                               254
Smartphone/Tablet ~ Handy                    234
Mobile ~ Smartphone                          212
Mobile ~ Mobile phone                        124
Electrical Item ~ Mobile                     108
Mobile ~ Mobile                               72
Smartphone/Tablet ~ Mobiltelefon              60
Mobile ~ Android Smartphone                   49
Misc ~ Telephone                              48
Mobile ~ Cell Phone                           37
Name: count, dtype: int64

In [6]:
mobils["marca"].value_counts()

marca
Desconeguda    1918
Apple          1371
Samsung        1020
Altres          615
Huawei          190
Nokia           133
Motorola        126
Xiaomi          112
Sony             94
Fairphone        73
Name: count, dtype: int64

## Desar les dades netes

No es pugen a git: hereten la llicència CC BY-SA 4.0 i es regeneren amb aquest quadern.

In [7]:
comu.PROCESSADES.mkdir(parents=True, exist_ok=True)
columnes = ["id", "data_provider", "country", "group_identifier", "any_event", "marca",
            "edat", "edat_neta", "repair_status", "final_vida",
            "repair_barrier_if_end_of_life", "partner_product_category", "problem"]
mobils[columnes].to_csv(comu.PROCESSADES / "mobils.csv", index=False)
print(f"Desat: {len(mobils):,} registres")

Desat: 5,652 registres
